In [1]:
import re
import numpy as np
import pandas as pd
import nltk 
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split



In [2]:
# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

#load datasets
try:
    train_df = pd.read_csv("train.csv", encoding="ISO-8859-1")
    test_df = pd.read_csv("test.csv", encoding="ISO-8859-1")
    print("Datasets loaded successfully!")
except FileNotFoundError:
    print("Please verify your file paths.")



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Datasets loaded successfully!


In [3]:
#Mapping column names
TEXT_COL = "text"
LABEL_COL = "sentiment"

#Drop rows with missing values
train_df = train_df.dropna(subset=[TEXT_COL, LABEL_COL])
train_df.head(10)


,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26
5,28b57f3990,http://www.dothebouncy.com/smf - some shameles...,http://www.dothebouncy.com/smf - some shameles...,neutral,night,70-100,Antigua and Barbuda,97929,440.0,223
6,6e0c6d75b1,2am feedings for the baby are fun when he is a...,fun,positive,morning,0-20,Argentina,45195774,2736690.0,17
7,50e14c0bb8,Soooo high,Soooo high,neutral,noon,21-30,Armenia,2963243,28470.0,104
8,e050245fbd,Both of you,Both of you,neutral,night,31-45,Australia,25499884,7682300.0,3
9,fc2cbefa9d,Journey!? Wow... u just became cooler. hehe....,Wow... u just became cooler.,positive,morning,46-60,Austria,9006398,82400.0,109


In [4]:
#PREPROCESSING FUNCTION
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
    # Remove URLs, handles, and special characters
    text = re.sub(r"http\S+|www\S+|https\S+|@\w+|[^a-zA-Z\s]", "", str(text), flags=re.MULTILINE)
    return text.lower()

train_df['cleaned_text'] = train_df['text'].apply(clean_tweet)
test_df['cleaned_text'] = test_df['text'].apply(clean_tweet)

In [5]:
#TF-IDF Vectorization
#We include basic English stopwords directly in the vectorizer here to save time
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
test_df = test_df.dropna(subset=["text", "sentiment"])

X_train = vectorizer.fit_transform(train_df['cleaned_text'])
X_test = vectorizer.transform(test_df['cleaned_text'])

y_train = train_df['sentiment']
y_test = test_df['sentiment']

In [6]:
#Training & Model Evaluation
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print("\nClassification Report:\n", classification_report(y_test, preds))


Accuracy: 0.6862

Classification Report:
               precision    recall  f1-score   support

    negative       0.71      0.61      0.66      1001
     neutral       0.62      0.73      0.67      1430
    positive       0.78      0.70      0.73      1103

    accuracy                           0.69      3534
   macro avg       0.70      0.68      0.69      3534
weighted avg       0.69      0.69      0.69      3534

